## Evaluation for code generation:

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
LLM_name = os.getenv('LLM_NAME')
LLM_name

'gpt-4.1-mini'

## LM-Eval
for GSM8K. Not found helpful for HumanEval

In [2]:
# Install the dependencies
if not os.path.exists('lm-evaluation-harness'):
	!git clone https://github.com/EleutherAI/lm-evaluation-harness.git
	!cd lm-evaluation-harness && pip install -e .[api]  # OpenAI API extras

In [ ]:
import lm_eval
import lm_eval.api.registry as registry
from lm_eval.api.registry import register_model
from lm_eval.models.openai_completions import OpenAIChatCompletion
import json

model_name = f'completions-custom'
# Delete the model if it already exists in the registry
if model_name in registry.MODEL_REGISTRY:
	del registry.MODEL_REGISTRY[model_name]

# Register a custom model that uses the OpenAI API with our own values
@register_model(model_name)
class CustomCompletionsAPI(OpenAIChatCompletion):
	def __init__(self, **kwargs):
		base_url = os.getenv('OPENAI_BASE_URL')
		if base_url:  # use base URL if provided
			kwargs['base_url'] = base_url + '/chat/completions'
		super().__init__(
			**kwargs,
			api_key=os.getenv('OPENAI_API_KEY'),
			model=LLM_name,
			tokenizer_backend='huggingface',
			tokenizer='gpt2',
			use_fast_tokenizer=True,
		)

task = 'gsm8k'  # 'gsm8k', 'humaneval', etc.
results = lm_eval.simple_evaluate(
	model=model_name,
	tasks=[task],
	batch_size=1,
	num_fewshot=0,
	# start=0,
	# limit=30,
	apply_chat_template=True,  # because we use a chat model, not a completions model
	confirm_run_unsafe_code=True,
)


# Write the results to a file

# Some responses fail to get converted to JSON. This avoids such errors.
def parse_obj(obj):
	if isinstance(obj, (str, int, float, bool, type(None))):
		return obj
	elif isinstance(obj, (list, tuple)):
		return [parse_obj(item) for item in obj]
	elif isinstance(obj, dict):
		return { str(key): parse_obj(value) for key, value in obj.items() }
	else:  # Convert non-serializable objects to their string representation
		return str(obj)

result_file = os.path.join('eval_results', f'{task}_{LLM_name}.json')
results_data = parse_obj(results)
with open(result_file, 'w') as file:
	json.dump(results_data, file, indent=4)
print(f'\nResults saved to {result_file} \n')
print(results_data['results'][task])

model_args not specified. Using defaults.
chat-completions endpoint requires the `--apply_chat_template` flag.
Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Requesting API: 100%|██████████| 30/30 [00:42<00:00,  1.42s/it]



Results saved to eval_results/humaneval_gemma2-9b-it.json 

{'alias': 'humaneval', 'pass@1,create_test': 0.0, 'pass@1_stderr,create_test': 0.0}


## EvalPlus
for HumanEval scores.

Documentation: https://github.com/evalplus/evalplus

In [ ]:
!pip install --upgrade evalplus

In [ ]:
# OpenAI models
!evalplus.evaluate --model "$LLM_name"  \
					--dataset humaneval   \
					--backend openai --greedy
# Loads OpenAI environment variables such as OPENAI_API_KEY, OPENAI_BASE_URL, etc.

Codegen: HumanEval/149 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━╺━━━ 148/164 • 0:00:08
Codegen: HumanEval/150 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━╺━━━ 149/164 • 0:00:10
Codegen: HumanEval/151 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━╸━━━ 150/164 • 0:00:11
Codegen: HumanEval/152 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━╸━━━ 151/164 • 0:00:13
Codegen: HumanEval/153 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━╺━━ 152/164 • 0:00:14
Codegen: HumanEval/154 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━╺━━ 153/164 • 0:00:16
Codegen: HumanEval/155 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━╸━━ 154/164 • 0:00:18
Codegen: HumanEval/156 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━╸━━ 155/164 • 0:00:19
Codegen: HumanEval/157 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━━╺━ 156/164 • 0:00:21
Codegen: HumanEval/158 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━━╺━ 157/164 • 0:00:22
Codegen: HumanEval/159 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━━╸━ 158/164 • 0:00:23
Codegen: HumanEval/160 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━━╸━ 159/164 • 0:00:25
Codegen: HumanEval/161 @ gemma2-9b-it━━━━━━━━━━━━━━━━━━━━━╺ 160/

## DeepEval
https://github.com/confident-ai/deepeval \
https://www.deepeval.com/docs/benchmarks-gsm8k

useful to evaluate using GSM8K and HumanEval

In [2]:
# from deepeval.models import LocalModel
# model = LocalModel()  # configured in ./deepeval/.deepeval file
# Failed to find accuracy using smaller models, so used OpenAI models.

from deepeval.benchmarks import GSM8K
from deepeval.models import GPTModel
model = GPTModel(model='gpt-4.1-mini', _openai_api_key=os.getenv('OPENAI_API_KEY'))

benchmark = GSM8K(n_problems=100)
benchmark.evaluate(model=model)
print(benchmark.overall_score)

Processing 100 problems: 100%|██████████| 100/100 [02:02<00:00,  1.23s/it]

Overall GSM8K Accuracy: 0.48
0.48


In [ ]:
%%writefile test_deepeval.py
# command: deepeval test run test_deepeval.py

# add code here and run the cell to store the code in the file

In [ ]:
!deepeval test run test_deepeval.py

## Result filter module - BeIR
https://github.com/beir-cellar/beir

To evaluate using MS MARCO

> code not written yet, not sure about why use this tool

In [ ]:
!pip install beir